In [1]:
import pymysql

try:
    conn = pymysql.connect(
        host="172.24.64.1",
        user="abhi",
        password="1289",
        database="customers",
        port=3306,
        connect_timeout=10
    )
    print("Connected!")
    #conn.close()
except Exception as e:
    print(type(e).__name__)
    print(e)

Connected!


In [2]:
import pandas as pd

query = "SELECT * FROM customer LIMIT 5;"

df = pd.read_sql(query, conn)

df

/tmp/ipykernel_17137/2797150005.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,09790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,01151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,08775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [3]:
from langchain_core.output_parsers import JsonOutputParser, StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
from all_api import groq_api

In [5]:
from langchain.chat_models import init_chat_model
from all_api import google_api

In [6]:
gem_llm=init_chat_model("google_genai:gemini-2.5-flash",api_key=google_api)

In [7]:
llm=ChatGroq(api_key=groq_api,model="llama-3.1-8b-instant")

In [8]:
response=llm.invoke("Hi")
print(response.content)

It's nice to meet you. Is there something I can help you with or would you like to chat?


In [9]:
schema="""
Database Schema with following tables

1. customer
   - customer_id
   - customer_unique_id
   - customer_zip_code_prefix
   - customer_city
   - customer_state

2. orders
   - order_id
   - customer_id
   - order_status
   - order_purchase_timestamp
   - order_approved_at
   - order_delivered_carrier_date
   - order_delivered_customer_date
   - order_estimated_delivery_date

3. order_items
   - order_id
   - order_item_id
   - product_id
   - seller_id
   - shipping_limit_date
   - price
   - freight_value

4. order_payments
   - order_id
   - payment_sequential
   - payment_type
   - payment_installments
   - payment_value

5. products
   - product_id
   - product_category_name
   - product_name_lenght
   - product_description_lenght
   - product_photos_qty
   - product_weight_g
   - product_length_cm
   - product_height_cm
   - product_width_cm
"""

In [10]:
query="What is the total revenue and average freight value for orders from customers in the state of São Paulo (SP), broken down by product category?"

#query="List the top 5 customers by total spending and the top 5 product categories by revenue."

#query="Which state generated the highest revenue, and which city has the most customers?"

In [11]:
prompt = """
You are an expert SQL analyst.

Your task is to answer the user's question by generating the most appropriate execution plan.

{schema}

Instructions:

General Rules:
- Generate only valid MySQL SQL.
- Use ONLY the tables and columns listed in the schema.
- Never invent tables or columns.
- Use appropriate JOINs only when required.
- Do not include explanations or markdown.
- Use null instead of None.

SQL Rules:
- Do NOT add LIMIT unless explicitly requested or implied by words such as "top", "highest", "lowest", "first", etc.
- Use COUNT(), SUM(), AVG(), MIN(), MAX(), GROUP BY, ORDER BY as required by the user's intent.

Data Notes:
- customer_state stores two-letter Brazilian state abbreviations (SP, RJ, MG, etc.).
- customer_city stores city names in lowercase.

Execution Planning Rules:

1. If the user's question can be answered with a single SQL query, return ONE query.

2. If the user asks multiple independent questions
   (e.g. "highest revenue state and most used payment method"),
   generate MULTIPLE independent SQL queries.

3. Do NOT combine unrelated analyses using UNION, artificial columns,
   or cross joins merely to force a single SQL query.

4. Each SQL query should answer exactly one analytical task.

5. Name every query descriptively.

User Question:
{query}

Return ONLY valid JSON not in markdown

Single-query example:

{{"plan_type": "single",
  "queries": [
    {{"name": "highest_revenue_state", "sql": "SELECT ..."}}
  ]}}

Multiple-query example:

{{"plan_type": "multiple",
  "queries": [
    {{"name": "highest_revenue_state", "sql": "SELECT ..."}},
    {{"name": "most_used_payment_method", "sql": "SELECT ..."}}
  ]}}
"""

In [12]:
print(prompt)


You are an expert SQL analyst.

Your task is to answer the user's question by generating the most appropriate execution plan.

{schema}

Instructions:

General Rules:
- Generate only valid MySQL SQL.
- Use ONLY the tables and columns listed in the schema.
- Never invent tables or columns.
- Use appropriate JOINs only when required.
- Do not include explanations or markdown.
- Use null instead of None.

SQL Rules:
- Do NOT add LIMIT unless explicitly requested or implied by words such as "top", "highest", "lowest", "first", etc.
- Use COUNT(), SUM(), AVG(), MIN(), MAX(), GROUP BY, ORDER BY as required by the user's intent.

Data Notes:
- customer_state stores two-letter Brazilian state abbreviations (SP, RJ, MG, etc.).
- customer_city stores city names in lowercase.

Execution Planning Rules:

1. If the user's question can be answered with a single SQL query, return ONE query.

2. If the user asks multiple independent questions
   (e.g. "highest revenue state and most used payment meth

In [13]:
parser = JsonOutputParser()
plan_prompt = ChatPromptTemplate.from_template(prompt)
plan_chain = plan_prompt | gem_llm | parser
response = plan_chain.invoke({"query": query, "schema": schema})
response

{'plan_type': 'single',
 'queries': [{'name': 'revenue_and_freight_by_product_category_for_SP',
   'sql': "SELECT p.product_category_name, SUM(oi.price) AS total_revenue, AVG(oi.freight_value) AS average_freight_value FROM customer AS c JOIN orders AS o ON c.customer_id = o.customer_id JOIN order_items AS oi ON o.order_id = oi.order_id JOIN products AS p ON oi.product_id = p.product_id WHERE c.customer_state = 'SP' GROUP BY p.product_category_name ORDER BY p.product_category_name;"}]}

In [14]:
type(response)

dict

In [15]:
response["queries"]

[{'name': 'revenue_and_freight_by_product_category_for_SP',
  'sql': "SELECT p.product_category_name, SUM(oi.price) AS total_revenue, AVG(oi.freight_value) AS average_freight_value FROM customer AS c JOIN orders AS o ON c.customer_id = o.customer_id JOIN order_items AS oi ON o.order_id = oi.order_id JOIN products AS p ON oi.product_id = p.product_id WHERE c.customer_state = 'SP' GROUP BY p.product_category_name ORDER BY p.product_category_name;"}]

In [ ]:
validation_template = """
You are an expert SQL reviewer.

Your job is to determine whether the generated SQL correctly answers the user's question.

Database Schema:
{schema}

User Question:
{query}

Generated SQL:
{generated_sql}

Validation Checklist:

1. Intent Validation
- Does the SQL match the user's intent?
- Examples:
  - "count", "how many", "number of" → COUNT()
  - "average", "mean" → AVG()
  - "total" → SUM()
  - "highest", "top" → ORDER BY DESC
  - "lowest" → ORDER BY ASC
  - "list", "show", "display" → SELECT rows

2. Schema Validation
- Only use tables present in the schema.
- Only use columns present in the schema.
- Do not invent tables or columns.

3. Join Validation
- Are the required joins present?
- Are unnecessary joins avoided?

4. Filter Validation
- Are all filters requested by the user applied?
- Are there any extra filters not requested?

5. Aggregation Validation
- Are GROUP BY, COUNT, SUM, AVG, MIN, MAX used correctly?
- Are aggregates missing when required?

6. Sorting & Limiting
- Is ORDER BY present when required?
- Is LIMIT used only when explicitly requested (e.g., "top 10")?

7. Semantic Validation
- Does executing this SQL answer the user's question?
- If the SQL returns a different result than requested, mark it invalid.

8. Database Manipulation
- Does the queries contain database modifying queries like UPDATE, DROP, ALTER, INSERT, DELETE. if found immediately return false only SELECT / WITH is allowed.

Respond ONLY in valid JSON.

{{"valid": "true","reason": "<one sentence explaining your decision>"}}

or

{{"valid": "false","reason": "<why the SQL does not answer the question>"}}
"""

validation_parser = JsonOutputParser()
validation_prompt_template = ChatPromptTemplate.from_template(validation_template)
validation_chain = validation_prompt_template | llm | validation_parser
val = validation_chain.invoke({"schema": schema, "query": query, "generated_sql": response["queries"]})

In [17]:
val

{'valid': 'true',
 'reason': "The SQL query correctly answers the user's question by filtering orders from customers in São Paulo, grouping by product category, and calculating total revenue and average freight value."}

In [18]:
print(val["valid"])

true


In [39]:
response["queries"]

[{'name': 'total_revenue_and_avg_freight_by_product_category_SP',
  'sql': "SELECT p.product_category_name, SUM(oi.price) AS total_revenue, AVG(oi.freight_value) AS average_freight_value FROM customer c JOIN orders o ON c.customer_id = o.customer_id JOIN order_items oi ON o.order_id = oi.order_id JOIN products p ON oi.product_id = p.product_id WHERE c.customer_state = 'SP' GROUP BY p.product_category_name ORDER BY p.product_category_name;"}]

In [40]:
for x in response["queries"]:
    print(x)

{'name': 'total_revenue_and_avg_freight_by_product_category_SP', 'sql': "SELECT p.product_category_name, SUM(oi.price) AS total_revenue, AVG(oi.freight_value) AS average_freight_value FROM customer c JOIN orders o ON c.customer_id = o.customer_id JOIN order_items oi ON o.order_id = oi.order_id JOIN products p ON oi.product_id = p.product_id WHERE c.customer_state = 'SP' GROUP BY p.product_category_name ORDER BY p.product_category_name;"}


In [19]:
FORBIDDEN = [
    "UPDATE",
    "DELETE",
    "DROP",
    "ALTER",
    "INSERT",
    "TRUNCATE",
    "CREATE",
    "REPLACE"
]

In [ ]:
results={}
retry_response={}
if val["valid"]=="true":
    for x in response["queries"]:
        try:
            if any(word in x["sql"].upper() for word in FORBIDDEN):
                raise Exception("Unsafe SQL")
            else:
                results[x["name"]]=pd.read_sql(x["sql"],conn).to_markdown()
        except Exception as e:
            print(e)
elif val["valid"]!="true":
    max_retries=0
    k=val["valid"]
    res=val["reason"]
    retry_prompt= "Previous Query was failed due to {reason}. Generate new query"+prompt
    retry_prompt=ChatPromptTemplate.from_template(retry_prompt)
    retry_chain=retry_prompt | gem_llm | parser
    while k!="true" and max_retries<2:
        retry_response= retry_chain.invoke({"schema":schema,"query":query,"reason":res})
        val = validation_chain.invoke({"schema": schema, "query": query, "generated_sql": retry_response["queries"]})
        k=val["valid"]
        res=val["reason"]
        max_retries+=1 
    
    if k=="true":
        for x in retry_response["queries"]:
            try:
                if any(word in x["sql"].upper() for word in FORBIDDEN):
                    raise Exception("Unsafe SQL")
                else:
                    results[x["name"]]=pd.read_sql(x["sql"],conn).to_markdown()
            except Exception as e:
                print(e)
    else:
        print("Invalid SQL query generated and retries exceeded")

/tmp/ipykernel_5113/3221946969.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  results[x["name"]]=pd.read_sql(x["sql"],conn).to_markdown()


In [42]:
results

{'total_revenue_and_avg_freight_by_product_category_SP': '|    | product_category_name                          |   total_revenue |   average_freight_value |\n|---:|:-----------------------------------------------|----------------:|------------------------:|\n|  0 | agro_industria_e_comercio                      |        25884.1  |                20.0572  |\n|  1 | alimentos                                      |        15478.6  |                10.4871  |\n|  2 | alimentos_bebidas                              |         6402.16 |                13.6303  |\n|  3 | artes                                          |        14937.8  |                15.5806  |\n|  4 | artes_e_artesanato                             |          906.25 |                11.5325  |\n|  5 | artigos_de_festas                              |         1585.88 |                14.3361  |\n|  6 | artigos_de_natal                               |         2143.98 |                13.021   |\n|  7 | audio                     

In [43]:
query

'What is the total revenue and average freight value for orders from customers in the state of São Paulo (SP), broken down by product category?'

In [ ]:
final_prompt_template = """
You are an expert Data Analyst.

Your task is to answer the user's question using ONLY the query results provided below.

User Question:
{query}

Query Results:
{results}

Instructions:
- Base your answer only on the provided query results.
- Do not make assumptions or invent information.
- If multiple query results are provided, combine them into a single coherent answer.
- Clearly reference the relevant metrics from each result.
- If the results are insufficient to answer the question, state what information is missing.
- Present the answer in clear business language suitable for a stakeholder.
- Do not mention SQL or the underlying database.

Final Answer:
"""

In [45]:
final_prompt = ChatPromptTemplate.from_template(final_prompt_template)
final_chain = final_prompt | llm
final_response = final_chain.invoke({"query": query, "results": results})
final_response.content

"Based on the provided query results, here are the total revenue and average freight value for orders from customers in the state of São Paulo (SP), broken down by product category:\n\n- The total revenue from orders in São Paulo for the 'agro_industria_e_comercio' category is R$ 25,884.10, with an average freight value of R$ 20.06.\n- For the 'alimentos' category, the total revenue is R$ 15,478.60, with an average freight value of R$ 10.49.\n- The 'alimentos_bebidas' category generated R$ 6,402.16 in total revenue, with an average freight value of R$ 13.63.\n- The 'artes' category had a total revenue of R$ 14,937.80, with an average freight value of R$ 15.58.\n- For the 'artes_e_artesanato' category, the total revenue is R$ 906.25, with an average freight value of R$ 11.53.\n- The 'artigos_de_festas' category generated R$ 1,585.88 in total revenue, with an average freight value of R$ 14.34.\n- The 'artigos_de_natal' category had a total revenue of R$ 2,143.98, with an average freight 

In [46]:
print(final_response.content)

Based on the provided query results, here are the total revenue and average freight value for orders from customers in the state of São Paulo (SP), broken down by product category:

- The total revenue from orders in São Paulo for the 'agro_industria_e_comercio' category is R$ 25,884.10, with an average freight value of R$ 20.06.
- For the 'alimentos' category, the total revenue is R$ 15,478.60, with an average freight value of R$ 10.49.
- The 'alimentos_bebidas' category generated R$ 6,402.16 in total revenue, with an average freight value of R$ 13.63.
- The 'artes' category had a total revenue of R$ 14,937.80, with an average freight value of R$ 15.58.
- For the 'artes_e_artesanato' category, the total revenue is R$ 906.25, with an average freight value of R$ 11.53.
- The 'artigos_de_festas' category generated R$ 1,585.88 in total revenue, with an average freight value of R$ 14.34.
- The 'artigos_de_natal' category had a total revenue of R$ 2,143.98, with an average freight value of 